# Tool calling mode
Turn the Pydantic schema into the parameters of a "fake" function and force the model to call it.

In [1]:
import json
import time

from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import JSON, Markdown, display

load_dotenv()  # loads GEMINI_API_KEY from .env
client = genai.Client()

from demo_helpers import VALID_APPLICATIONS

In [2]:
transcript = "Hey Google, navigate to Ara."
TOOL_NAME = "submit_parsed_command"

In [3]:
from pydantic import BaseModel, Field


class ParsedCommand(BaseModel):
    target_application: str = Field(description="The application the user wants to use")
    target_action: str = Field(
        description="The action to perform within that application"
    )
    additional_parameters: dict[str, str] = Field(
        default_factory=dict,
        description=(
            "Parameters for the action mentioned in the transcript, as a mapping "
            "of parameter name (as listed for the chosen action) to value"
        ),
    )


print(json.dumps(ParsedCommand.model_json_schema(), indent=2))

{
  "properties": {
    "target_application": {
      "description": "The application the user wants to use",
      "title": "Target Application",
      "type": "string"
    },
    "target_action": {
      "description": "The action to perform within that application",
      "title": "Target Action",
      "type": "string"
    },
    "additional_parameters": {
      "additionalProperties": {
        "type": "string"
      },
      "description": "Parameters for the action mentioned in the transcript, as a mapping of parameter name (as listed for the chosen action) to value",
      "title": "Additional Parameters",
      "type": "object"
    }
  },
  "required": [
    "target_application",
    "target_action"
  ],
  "title": "ParsedCommand",
  "type": "object"
}


## Input

In [10]:
message = f"""For the following transcript, determine the target action the user wants to take,
the target application, and any additional parameters.\n
The target application, action and parameters must be chosen from the following
valid applications (each lists its actions and the parameters each action accepts):\n
{json.dumps(VALID_APPLICATIONS, indent=2)}\n
Submit your answer by calling the {TOOL_NAME} tool.

transcript: {transcript}"""
print(message)

For the following transcript, determine the target action the user wants to take,
the target application, and any additional parameters.

The target application, action and parameters must be chosen from the following
valid applications (each lists its actions and the parameters each action accepts):

{
  "maps": {
    "description": "Navigation and location search",
    "actions": {
      "navigate": {
        "destination": "str",
        "mode": "driving | walking | cycling | transit"
      },
      "search_nearby": {
        "query": "str",
        "radius_km": "float"
      },
      "show_traffic": {}
    }
  },
  "music": {
    "description": "Music streaming and playback",
    "actions": {
      "play": {
        "song": "str",
        "artist": "str",
        "playlist": "str"
      },
      "pause": {},
      "skip": {},
      "set_volume": {
        "level": "int (0-100)"
      }
    }
  },
  "messages": {
    "description": "Send and read text messages",
    "actions": {
   

In [5]:
submit_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name=TOOL_NAME,
            description="Submit the parsed command extracted from the transcript.",
            parameters_json_schema=ParsedCommand.model_json_schema(),
        )
    ]
)

config = types.GenerateContentConfig(
    tools=[submit_tool],
    # Force the model to call our tool (no free-text answer allowed)
    tool_config=types.ToolConfig(
        function_calling_config=types.FunctionCallingConfig(
            mode="ANY",
            allowed_function_names=[TOOL_NAME],
        )
    ),
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)
config

GenerateContentConfig(
  automatic_function_calling=AutomaticFunctionCallingConfig(
    disable=True,
    maximum_remote_calls=10
  ),
  tool_config=ToolConfig(
    function_calling_config=FunctionCallingConfig(
      allowed_function_names=[
        'submit_parsed_command',
      ],
      mode=<FunctionCallingConfigMode.ANY: 'ANY'>
    )
  ),
  tools=[
    Tool(
      function_declarations=[
        FunctionDeclaration(
          description='Submit the parsed command extracted from the transcript.',
          name='submit_parsed_command',
          parameters_json_schema={
            'properties': {<... 3 items at Max depth ...>},
            'required': [<... 2 items at Max depth ...>],
            'title': 'ParsedCommand',
            'type': 'object'
          }
        ),
      ]
    ),
  ]
)

## Model call

In [11]:
start = time.perf_counter()
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=message,
    config=config,
)
print(f"Client call took {time.perf_counter() - start:.2f}s")

Client call took 1.33s


## Output

In [12]:
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'additional_parameters': {<... 1 item at Max depth ...>},
            'target_action': 'navigate',
            'target_application': 'maps'
          },
          id='call_18203683053412974594',
          name='submit_parsed_command'
        ),
        thought_signature=b'\x12`\n^\x01i\x14}\x13\xfc;\xbd\xe9\xed\xea\xc2\x04\xac\xf8\xe9y,\xa9gJ\t\xfaYZS6R\xbe\xc8\xac\x9cdc\xfa\xf5\x0e7uQ\xbb{\n\xc7\xdayD\x0c\xafy\xf9\xd6\xe7\x06O\xd9b+"\xfeFs\xc1\xe7\x14\xf5t\x113:\x95\xe2Y\xabJ\x86\x08\xda\xa2t\xe0\x86-c\xbf\xb2\xc0M%\x01+/!\xe1'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.5-flash-lite' prompt_feedback=None response_id='qq_6aonXINPZjNAPuIKWYA' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_t

In [13]:
function_call = response.function_calls[0]
function_call

FunctionCall(
  args={
    'additional_parameters': {
      'destination': 'Ara'
    },
    'target_action': 'navigate',
    'target_application': 'maps'
  },
  id='call_18203683053412974594',
  name='submit_parsed_command'
)

In [14]:
# Validate the tool-call arguments against the Pydantic model
result = ParsedCommand.model_validate(function_call.args)
result

ParsedCommand(target_application='maps', target_action='navigate', additional_parameters={'destination': 'Ara'})